In [ ]:
# Dependencies are installed from task/task3_1/requirements.txt.


In [ ]:
# The PathMNIST-derived subset is provided under dataset/task3_1/.


# MedMNIST
Это коллекция из 18 медицинских датасетов, предварительно обработанных до стандартизированного формата (28×28 пикселей для 2D, 28×28×28 для 3D), аналогичного классическому MNIST. Проект предназначен для быстрого прототипирования, обучения и бенчмаркинга моделей машинного обучения на медицинских изображениях.

## 2D датасеты MedMNIST (28×28 пикселей)

| Флаг (class) | Название | Модальность | Задача | Классов | Изображений | Каналы |
|:-------------|:----------|:------------|:-------|--------:|------------:|-------:|
| `pathmnist` | PathMNIST | Гистопатология (лимфоузлы, рак молочной железы) | Многоклассовая | 9 | ~107,000 | 3 (RGB) |
| `octmnist` | OCTMNIST | Оптическая когерентная томография (сетчатка) | Многоклассовая | 4 | ~109,000 | 1 |
| `pneumoniamnist` | PneumoniaMNIST | Рентген грудной клетки | Бинарная | 2 | ~5,856 | 1 |
| `chestmnist` | ChestMNIST | Рентген грудной клетки | Мультиметка (14 патологий) | 14 | ~112,000 | 1 |
| `dermamnist` | DermaMNIST | Дерматоскопия (кожные поражения) | Многоклассовая | 7 | ~10,015 | 3 (RGB) |
| `retinamnist` | RetinaMNIST | Фундус-фотография (диабетическая ретинопатия) | Многоклассовая | 5 | ~1,600 | 3 (RGB) |
| `breastmnist` | BreastMNIST | УЗИ молочной железы | Многоклассовая | 3 | ~1,200 | 1 |
| `bloodmnist` | BloodMNIST | Микроскопия крови (типы лейкоцитов) | Многоклассовая | 8 | ~17,000 | 3 (RGB) |
| `tissuemnist` | TissueMNIST | Микроскопия биопсии почек | Многоклассовая | 8 | ~236,000 | 1 |
| `organamnist` | OrganAMNIST | КТ-срезы (осевая проекция) | Многоклассовая | 11 | ~58,000 | 1 |
| `organcmnist` | OrganCMNIST | КТ-срезы (корональная проекция) | Многоклассовая | 11 | ~58,000 | 1 |
| `organsmnist` | OrganSMNIST | КТ-срезы (сагиттальная проекция) | Многоклассовая | 11 | ~58,000 | 1 |



## 3D датасеты MedMNIST (28×28×28 вокселей)

| Флаг (class) | Название | Модальность | Задача | Классов | Образцов | Каналы |
|:-------------|:----------|:------------|:-------|--------:|---------:|-------:|
| `organmnist3d` | OrganMNIST3D | Компьютерная томография (сегментированные органы) | Многоклассовая | 11 | ~2,800 | 1 |
| `nodulemnist3d` | NoduleMNIST3D | КТ (легочные узелки) | Бинарная | 2 | ~1,600 | 1 |
| `adrenalmnist3d` | AdrenalMNIST3D | КТ (надпочечники) | Бинарная | 2 | ~1,300 | 1 |
| `fracturemnist3d` | FractureMNIST3D | КТ (позвоночник, переломы) | Бинарная | 2 | ~1,000 | 1 |
| `vesselmnist3d` | VesselMNIST3D | Магнитно-резонансная ангиография (сосуды мозга) | Бинарная | 2 | ~1,300 | 1 |
| `synapsemnist3d` | SynapseMNIST3D | Электронная микроскопия (синапсы) | Бинарная | 2 | ~1,500 | 1 |



## Задачи

| Тип задачи | 2D датасеты | 3D датасеты |
|:-----------|:------------|:------------|
| **Бинарная классификация** | PneumoniaMNIST | NoduleMNIST3D, AdrenalMNIST3D, FractureMNIST3D, VesselMNIST3D, SynapseMNIST3D |
| **Многоклассовая классификация** | PathMNIST, OCTMNIST, DermaMNIST, RetinaMNIST, BreastMNIST, BloodMNIST, TissueMNIST, OrganA/C/SMNIST | OrganMNIST3D |
| **Мультиметка (14 классов)** | ChestMNIST | — |


In [ ]:
from pathlib import Path

from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


# 28 X 28

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

NUM_EPOCHS = 10
BATCH_SIZE = 128
lr = 0.001
n_channels = 3
n_classes = 9
task = 'multi-class'


In [ ]:
from pathlib import Path
import io
from urllib.request import urlopen

repo_root = Path.cwd()
if repo_root.name in {'vanilla-code', 'your-sollution', 'your_solution'}:
    repo_root = repo_root.parent

def dataset_path(task_name: str, filename: str) -> str:
    local_path = repo_root / "dataset" / task_name / filename
    if local_path.exists():
        return str(local_path)
    return (
        'https://github.com/AI-is-out-there/'
        'check-your-biomed-datascience-skills-review/raw/refs/heads/dev/'
        f'dataset/{task_name}/{filename}'
    )

def load_npz(path):
    if str(path).startswith("http"):
        with urlopen(path) as response:
            return np.load(io.BytesIO(response.read()), allow_pickle=False)
    return np.load(path, allow_pickle=False)

train_path = dataset_path('task3_1', 'train.npz')
test_features_path = dataset_path('task3_1', 'test_features.npz')

train_data = load_npz(train_path)
test_data = load_npz(test_features_path)

train_images = train_data['images']
train_targets = train_data['prediction']
label_names = train_data['label_names']
test_images = test_data['images']
test_image_ids = test_data['image_id']

image_tensors = torch.from_numpy(train_images).permute(0, 3, 1, 2).float() / 255.0
target_tensors = torch.from_numpy(train_targets).long()
test_tensors = torch.from_numpy(test_images).permute(0, 3, 1, 2).float() / 255.0
indices = np.arange(len(train_targets))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=train_targets
)
train_dataset = data.TensorDataset(image_tensors[train_idx], target_tensors[train_idx])
val_dataset = data.TensorDataset(image_tensors[val_idx], target_tensors[val_idx])
train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader_at_eval = data.DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = data.DataLoader(test_tensors, batch_size=BATCH_SIZE)


In [ ]:
print('train:', len(train_dataset), 'validation:', len(val_dataset), 'held out:', len(test_images))
print('image shape:', train_images.shape[1:], 'classes:', list(label_names))


In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(18, 2.5))
for class_id, ax in enumerate(axes):
    image = train_images[np.flatnonzero(train_targets == class_id)[0]]
    ax.imshow(image)
    ax.set_title(f'{class_id}: {label_names[class_id]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for ax, image, target in zip(axes.ravel(), train_images[:18], train_targets[:18]):
    ax.imshow(image)
    ax.set_title(f'{target}: {label_names[target]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()


In [ ]:
# define a simple CNN model

class Net(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(Net, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU())

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))

        self.layer3 = nn.Sequential(
            nn.Conv2d(16, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())
        
        self.layer4 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())

        self.layer5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))

        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = Net(in_channels=n_channels, num_classes=n_classes)
    
# define loss function and optimizer
if task == "multi-label, binary-class":
    criterion = nn.BCEWithLogitsLoss()
else:
    criterion = nn.CrossEntropyLoss()
    
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
for epoch in range(NUM_EPOCHS):
    train_correct = 0
    train_total = 0
    test_correct = 0
    test_total = 0
    
    model.train()
    for inputs, targets in tqdm(train_loader):
        # forward + backward + optimize
        optimizer.zero_grad()
        outputs = model(inputs)
        
        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32)
            loss = criterion(outputs, targets)
        else:
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()

In [ ]:
# evaluation on the public validation split

def evaluate(loader, split_name):
    model.eval()
    targets_all, predictions_all = [], []
    with torch.no_grad():
        for inputs, targets in loader:
            predictions = model(inputs).argmax(dim=1)
            targets_all.extend(targets.numpy())
            predictions_all.extend(predictions.numpy())
    accuracy = accuracy_score(targets_all, predictions_all)
    macro_f1 = f1_score(targets_all, predictions_all, average='macro')
    print(f'{split_name} accuracy: {accuracy:.3f}  macro F1: {macro_f1:.3f}')
    return accuracy, macro_f1

print('==> Evaluating ...')
evaluate(train_loader_at_eval, 'train')
evaluate(val_loader, 'validation')

# predictions for the held-out images
model.eval()
held_out_predictions = []
with torch.no_grad():
    for inputs in test_loader:
        held_out_predictions.extend(model(inputs).argmax(dim=1).numpy())

output_path = repo_root / 'outputs' / 'task3_1_predictions.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({
    'image_id': test_image_ids,
    'prediction': held_out_predictions,
}).to_csv(output_path, index=False)
print(f'Predictions saved to {output_path}')


## Scope note

The former 224×224 and 3D MedMNIST demonstrations were removed because they require unrelated datasets and CUDA. The Task 3.1 teaching model remains the original 28×28 PathMNIST CNN.
